<a href="https://colab.research.google.com/github/garvagrawalhere/URL-Shortener/blob/main/URL_Shortener.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# URL SHORTENER

By Garv Agrawal

In [1]:
!apt-get update -qq
!apt-get install -y -qq postgresql postgresql-contrib redis-server

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
!pip install -q fastapi uvicorn psycopg2-binary redis pydantic requests locust nest_asyncio

###Starting and configuring Postgresql and Redis

In [3]:
import subprocess
import time

# Start PostgreSQL
subprocess.run(["service", "postgresql", "start"], check=True)

# Set PostgreSQL password
subprocess.run([
    "sudo", "-u", "postgres",
    "psql", "-c",
    "ALTER USER postgres PASSWORD 'postgres';"
], check=True)

# Start Redis
subprocess.run([
    "redis-server",
    "--daemonize", "yes"
], check=True)

time.sleep(2)

print("PostgreSQL started")
print("Redis started")

PostgreSQL started
Redis started


###Creating Database and Table

In [4]:
import psycopg2

# Connect to default PostgreSQL database
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="postgres",
    user="postgres",
    password="postgres"
)

conn.autocommit = True
cursor = conn.cursor()

# Create our database
cursor.execute(
    "SELECT 1 FROM pg_database WHERE datname = 'urlshortener'"
)

if cursor.fetchone() is None:
    cursor.execute("CREATE DATABASE urlshortener")
    print("✅ Database created")
else:
    print("Database already exists")

cursor.close()
conn.close()


# Connect to our project database
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="urlshortener",
    user="postgres",
    password="postgres"
)

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS urls (
    id BIGSERIAL PRIMARY KEY,
    short_id VARCHAR(20) UNIQUE NOT NULL,
    original_url TEXT NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    expires_at TIMESTAMP NULL,
    click_count BIGINT DEFAULT 0
);
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_urls_short_id
ON urls(short_id);
""")

conn.commit()

cursor.close()
conn.close()

print("✅ Database schema ready")

Database already exists
✅ Database schema ready


### Adding Base62 + Redis + database helpers

In [5]:
import psycopg2
import redis
import time

# -------------------------
# Redis
# -------------------------

redis_client = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

print("Redis:", redis_client.ping())


# -------------------------
# Base62
# -------------------------

BASE62 = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"


def base62_encode(number):
    if number == 0:
        return "0"

    result = []

    while number > 0:
        result.append(BASE62[number % 62])
        number //= 62

    return "".join(reversed(result))


def base62_decode(value):
    number = 0

    for char in value:
        number = number * 62 + BASE62.index(char)

    return number


print("12345 →", base62_encode(12345))
print("3D7 →", base62_decode("3D7"))


# -------------------------
# PostgreSQL helper
# -------------------------

def get_connection():
    return psycopg2.connect(
        host="localhost",
        port=5432,
        database="urlshortener",
        user="postgres",
        password="postgres"
    )

Redis: True
12345 → 3D7
3D7 → 12345


###Adding the Token bucket rate limiter

In [6]:
RATE_LIMIT = 100
REFILL_RATE = 100 / 60  # tokens per second


TOKEN_BUCKET_SCRIPT = redis_client.register_script("""
local key = KEYS[1]

local capacity = tonumber(ARGV[1])
local refill_rate = tonumber(ARGV[2])
local now = tonumber(ARGV[3])

local data = redis.call(
    "HMGET",
    key,
    "tokens",
    "last_refill"
)

local tokens = tonumber(data[1])
local last_refill = tonumber(data[2])

if tokens == nil then
    tokens = capacity
    last_refill = now
end

local elapsed = math.max(0, now - last_refill)

tokens = math.min(
    capacity,
    tokens + elapsed * refill_rate
)

local allowed = 0

if tokens >= 1 then
    tokens = tokens - 1
    allowed = 1
end

redis.call(
    "HMSET",
    key,
    "tokens", tokens,
    "last_refill", now
)

redis.call("EXPIRE", key, 120)

return {allowed, tokens}
""")


def allow_request(ip):

    result = TOKEN_BUCKET_SCRIPT(
        keys=[f"rate:{ip}"],
        args=[
            RATE_LIMIT,
            REFILL_RATE,
            time.time()
        ]
    )

    return bool(result[0])

###The main FastAPI app

In [7]:
%%writefile app.py

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import RedirectResponse
from pydantic import BaseModel, HttpUrl

import psycopg2
import redis
import time
from datetime import datetime, timedelta


app = FastAPI(
    title="High Performance URL Shortener"
)


# ============================================================
# CONNECTIONS
# ============================================================

redis_client = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)


def get_connection():

    return psycopg2.connect(
        host="localhost",
        port=5432,
        database="urlshortener",
        user="postgres",
        password="postgres"
    )


# ============================================================
# BASE62
# ============================================================

BASE62 = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"


def base62_encode(number):

    if number == 0:
        return "0"

    result = []

    while number > 0:
        result.append(BASE62[number % 62])
        number //= 62

    return "".join(reversed(result))


# ============================================================
# TOKEN BUCKET
# ============================================================

RATE_LIMIT = 100
REFILL_RATE = 100 / 60


TOKEN_BUCKET_SCRIPT = redis_client.register_script("""
local key = KEYS[1]

local capacity = tonumber(ARGV[1])
local refill_rate = tonumber(ARGV[2])
local now = tonumber(ARGV[3])

local data = redis.call(
    "HMGET",
    key,
    "tokens",
    "last_refill"
)

local tokens = tonumber(data[1])
local last_refill = tonumber(data[2])

if tokens == nil then
    tokens = capacity
    last_refill = now
end

local elapsed = math.max(0, now - last_refill)

tokens = math.min(
    capacity,
    tokens + elapsed * refill_rate
)

local allowed = 0

if tokens >= 1 then
    tokens = tokens - 1
    allowed = 1
end

redis.call(
    "HMSET",
    key,
    "tokens", tokens,
    "last_refill", now
)

redis.call("EXPIRE", key, 120)

return {allowed, tokens}
""")


def allow_request(ip):

    result = TOKEN_BUCKET_SCRIPT(
        keys=[f"rate:{ip}"],
        args=[
            RATE_LIMIT,
            REFILL_RATE,
            time.time()
        ]
    )

    return bool(result[0])


# ============================================================
# REQUEST MODEL
# ============================================================

class URLRequest(BaseModel):

    url: HttpUrl

    expires_in_seconds: int | None = None


# ============================================================
# HEALTH CHECK
# ============================================================

@app.get("/")
def home():

    return {
        "message": "URL Shortener is running"
    }


@app.get("/health")
def health():

    return {
        "status": "healthy",
        "redis": redis_client.ping()
    }


# ============================================================
# CREATE SHORT URL
# ============================================================

@app.post("/shorten")
def shorten(
    data: URLRequest,
    request: Request
):

    ip = request.client.host

    if not allow_request(ip):

        raise HTTPException(
            status_code=429,
            detail="Rate limit exceeded"
        )

    expires_at = None

    if data.expires_in_seconds is not None:

        if data.expires_in_seconds <= 0:

            raise HTTPException(
                status_code=400,
                detail="Expiry must be positive"
            )

        expires_at = (
            datetime.now()
            + timedelta(
                seconds=data.expires_in_seconds
            )
        )

    conn = get_connection()

    try:

        cursor = conn.cursor()

        # First insert using temporary short_id
        cursor.execute(
            """
            INSERT INTO urls
            (short_id, original_url, expires_at)
            VALUES (%s, %s, %s)
            RETURNING id
            """,
            (
                "TEMP",
                str(data.url),
                expires_at
            )
        )

        numeric_id = cursor.fetchone()[0]

        short_id = base62_encode(numeric_id)

        cursor.execute(
            """
            UPDATE urls
            SET short_id = %s
            WHERE id = %s
            """,
            (short_id, numeric_id)
        )

        conn.commit()

        return {
            "short_id": short_id,
            "short_url": f"/r/{short_id}",
            "original_url": str(data.url)
        }

    finally:

        cursor.close()
        conn.close()


# ============================================================
# REDIRECT
# ============================================================

@app.get("/r/{short_id}")
def redirect(
    short_id: str,
    request: Request
):

    ip = request.client.host

    if not allow_request(ip):

        raise HTTPException(
            status_code=429,
            detail="Rate limit exceeded"
        )


    # -------------------------
    # Redis cache
    # -------------------------

    cached_url = redis_client.get(
        f"url:{short_id}"
    )


    if cached_url:

        redis_client.incr(
            "metrics:cache_hits"
        )

        # Analytics
        conn = get_connection()

        try:

            cursor = conn.cursor()

            cursor.execute(
                """
                UPDATE urls
                SET click_count = click_count + 1
                WHERE short_id = %s
                """,
                (short_id,)
            )

            conn.commit()

        finally:

            cursor.close()
            conn.close()


        return RedirectResponse(
            cached_url,
            status_code=307
        )


    # -------------------------
    # Cache miss → PostgreSQL
    # -------------------------

    redis_client.incr(
        "metrics:cache_misses"
    )

    conn = get_connection()

    try:

        cursor = conn.cursor()

        cursor.execute(
            """
            SELECT
                original_url,
                expires_at
            FROM urls
            WHERE short_id = %s
            """,
            (short_id,)
        )

        result = cursor.fetchone()

        if result is None:

            raise HTTPException(
                status_code=404,
                detail="Short URL not found"
            )

        original_url, expires_at = result


        # Expiry
        if (
            expires_at is not None
            and datetime.now() > expires_at
        ):

            raise HTTPException(
                status_code=410,
                detail="Short URL expired"
            )


        # Update analytics
        cursor.execute(
            """
            UPDATE urls
            SET click_count = click_count + 1
            WHERE short_id = %s
            """,
            (short_id,)
        )

        conn.commit()

    finally:

        cursor.close()
        conn.close()


    # Put URL into Redis
    redis_client.setex(
        f"url:{short_id}",
        3600,
        original_url
    )


    return RedirectResponse(
        original_url,
        status_code=307
    )


# ============================================================
# BENCHMARK ENDPOINTS
# ============================================================

@app.get("/benchmark/postgres/{short_id}")
def benchmark_postgres(short_id: str):

    conn = get_connection()

    try:

        cursor = conn.cursor()

        cursor.execute(
            """
            SELECT original_url
            FROM urls
            WHERE short_id = %s
            """,
            (short_id,)
        )

        result = cursor.fetchone()

        if result is None:

            raise HTTPException(
                status_code=404
            )

        return {
            "url": result[0]
        }

    finally:

        cursor.close()
        conn.close()


@app.get("/benchmark/redis/{short_id}")
def benchmark_redis(short_id: str):

    cached_url = redis_client.get(
        f"url:{short_id}"
    )

    if cached_url is None:

        conn = get_connection()

        try:

            cursor = conn.cursor()

            cursor.execute(
                """
                SELECT original_url
                FROM urls
                WHERE short_id = %s
                """,
                (short_id,)
            )

            result = cursor.fetchone()

            if result is None:

                raise HTTPException(
                    status_code=404
                )

            cached_url = result[0]

        finally:

            cursor.close()
            conn.close()


        redis_client.setex(
            f"url:{short_id}",
            3600,
            cached_url
        )


    return {
        "url": cached_url
    }

Overwriting app.py


In [8]:
import subprocess
import time

server = subprocess.Popen(
    [
        "uvicorn",
        "app:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ]
)

time.sleep(3)

print("✅ FastAPI running")

✅ FastAPI running


###Basic API Testing

In [9]:
import requests

r = requests.get(
    "http://127.0.0.1:8000/health"
)

print(r.status_code)
print(r.json())

200
{'status': 'healthy', 'redis': True}


###Creating and testing a URL

In [10]:
payload = {
    "url": "https://www.youtube.com/",
    "expires_in_seconds": 3600
}

r = requests.post(
    "http://127.0.0.1:8000/shorten",
    json=payload
)

print(r.json())

short_id = r.json()["short_id"]

{'short_id': '3', 'short_url': '/r/3', 'original_url': 'https://www.youtube.com/'}


In [11]:
r = requests.get(
    f"http://127.0.0.1:8000/r/{short_id}",
    allow_redirects=False
)

print("Status:", r.status_code)
print("Redirect:", r.headers.get("location"))

Status: 307
Redirect: https://www.youtube.com/


###Testing Caching and Analytics

In [12]:
# Clear metrics
redis_client.set("metrics:cache_hits", 0)
redis_client.set("metrics:cache_misses", 0)

# First request → PostgreSQL → Redis
requests.get(
    f"http://127.0.0.1:8000/r/{short_id}",
    allow_redirects=False
)

# Following requests → Redis
for _ in range(10):

    requests.get(
        f"http://127.0.0.1:8000/r/{short_id}",
        allow_redirects=False
    )


hits = int(
    redis_client.get("metrics:cache_hits") or 0
)

misses = int(
    redis_client.get("metrics:cache_misses") or 0
)

print("Cache hits:", hits)
print("Cache misses:", misses)

if hits + misses:
    print(
        "Cache hit rate:",
        round(
            hits / (hits + misses) * 100,
            2
        ),
        "%"
    )

Cache hits: 11
Cache misses: 0
Cache hit rate: 100.0 %


###Benchmark URL

In [13]:
payload = {
    "url": "https://example.com/performance-test"
}

r = requests.post(
    "http://127.0.0.1:8000/shorten",
    json=payload
)

benchmark_id = r.json()["short_id"]

print("Benchmark ID:", benchmark_id)

# Warm Redis
redis_client.setex(
    f"url:{benchmark_id}",
    3600,
    "https://example.com/performance-test"
)

Benchmark ID: 4


/tmp/ipykernel_10388/1177137418.py:15: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  redis_client.setex(


True

###Locust

In [14]:
import requests

# ==============================
# SHORTEN A URL
# ==============================

long_url = input("Enter the URL you want to shorten: ")

# Automatically add https:// if user doesn't provide it
if not long_url.startswith(("http://", "https://")):
    long_url = "https://" + long_url

payload = {
    "url": long_url,
    "expires_in_seconds": 3600
}

r = requests.post(
    "http://127.0.0.1:8000/shorten",
    json=payload
)

if r.status_code != 200:

    print("\n❌ Invalid URL.")
    print("Please enter something like:")
    print("https://example.com")

else:

    data = r.json()

    print("\n✅ URL shortened successfully!")
    print("Short ID :", data["short_id"])
    print("Short URL:", data["short_url"])

    # ==============================
    # GET ORIGINAL URL
    # ==============================

    short_input = input(
        "\nEnter the shortened URL or short ID to get the original URL: "
    )

    if "/r/" in short_input:
        short_id = short_input.split("/r/")[-1]
    else:
        short_id = short_input

    r = requests.get(
        f"http://127.0.0.1:8000/r/{short_id}",
        allow_redirects=False
    )

    if r.status_code == 307:

        print("\n✅ Original URL:")
        print(r.headers.get("location"))

    elif r.status_code == 404:

        print("\n❌ Short URL not found.")

    elif r.status_code == 410:

        print("\n❌ This short URL has expired.")

    else:

        print("\n❌ Error:", r.text)

Enter the URL you want to shorten: pubg77.com

✅ URL shortened successfully!
Short ID : 5
Short URL: /r/5

Enter the shortened URL or short ID to get the original URL: /r/5

✅ Original URL:
https://pubg77.com/


In [15]:
%%writefile locust_redis.py

from locust import HttpUser, task


class RedisUser(HttpUser):

    @task
    def test_redis(self):

        self.client.get(
            "/benchmark/redis/BENCHMARK_ID",
            name="Redis"
        )

Overwriting locust_redis.py


In [16]:
for filename in [
    "locust_postgres.py",
    "locust_redis.py"
]:

    with open(filename, "r") as f:
        content = f.read()

    content = content.replace(
        "BENCHMARK_ID",
        benchmark_id
    )

    with open(filename, "w") as f:
        f.write(content)

print("✅ Benchmark files ready")

✅ Benchmark files ready


OK, Now using 50 concurrent users for 10 sec

In [17]:
!locust -f locust_postgres.py \
    --headless \
    --host http://127.0.0.1:8000 \
    --users 50 \
    --spawn-rate 50 \
    --run-time 10s \
    --csv=postgres \
    --only-summary

[2026-08-17 00:33:38,157] f2ce4439bd36/INFO/locust.main: Starting Locust 2.46.3
[2026-08-17 00:33:38,158] f2ce4439bd36/INFO/locust.main: Run time limit set to 10 seconds
[2026-08-17 00:33:38,159] f2ce4439bd36/INFO/locust.runners: Ramping to 50 users at a rate of 50.00 per second
[2026-08-17 00:33:38,163] f2ce4439bd36/INFO/locust.runners: All users spawned: {"PostgreSQLUser": 50} (50 total users)
[2026-08-17 00:33:47,913] f2ce4439bd36/INFO/locust.main: --run-time limit reached, shutting down
[2026-08-17 00:33:48,654] f2ce4439bd36/INFO/locust.main: Shutting down (exit code 0)
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
GET      PostgreSQL     466     0(0.00%) |    942     147    1862    860 |   45.54        0.00
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated     466     0(0.00%) |    942     147    1862

###Redis Benchmark

In [18]:
!locust -f locust_redis.py \
    --headless \
    --host http://127.0.0.1:8000 \
    --users 50 \
    --spawn-rate 50 \
    --run-time 10s \
    --csv=redis \
    --only-summary

[2026-08-17 00:33:50,885] f2ce4439bd36/INFO/locust.main: Starting Locust 2.46.3
[2026-08-17 00:33:50,885] f2ce4439bd36/INFO/locust.main: Run time limit set to 10 seconds
[2026-08-17 00:33:50,886] f2ce4439bd36/INFO/locust.runners: Ramping to 50 users at a rate of 50.00 per second
[2026-08-17 00:33:50,890] f2ce4439bd36/INFO/locust.runners: All users spawned: {"RedisUser": 50} (50 total users)
[2026-08-17 00:34:00,630] f2ce4439bd36/INFO/locust.main: --run-time limit reached, shutting down
[2026-08-17 00:34:00,849] f2ce4439bd36/INFO/locust.main: Shutting down (exit code 0)
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
GET      Redis    5195     0(0.00%) |     73      26     617     64 |  522.92        0.00
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated    5195     0(0.00%) |     73      26     617     64 | 

##Comparing With and without Redis' Caching

In [19]:
import pandas as pd

pg = pd.read_csv("postgres_stats.csv")
rd = pd.read_csv("redis_stats.csv")

pg = pg.iloc[0]
rd = rd.iloc[0]

pg_latency = pg["Average Response Time"]
rd_latency = rd["Average Response Time"]

pg_rps = pg["Requests/s"]
rd_rps = rd["Requests/s"]

latency_reduction = (
    (pg_latency - rd_latency)
    / pg_latency
) * 100

rps_improvement = (
    (rd_rps - pg_rps)
    / pg_rps
) * 100

print("========== RESULTS ==========")

print(
    f"PostgreSQL latency : {pg_latency:.2f} ms"
)

print(
    f"Redis latency      : {rd_latency:.2f} ms"
)

print(
    f"Latency reduction  : {latency_reduction:.2f}%"
)

print()

print(
    f"PostgreSQL RPS     : {pg_rps:.2f}"
)

print(
    f"Redis RPS          : {rd_rps:.2f}"
)

print(
    f"RPS improvement    : {rps_improvement:.2f}%"
)

========== RESULTS ==========
PostgreSQL latency : 934.92 ms
Redis latency      : 69.06 ms
Latency reduction  : 92.61%

PostgreSQL RPS     : 44.82
Redis RPS          : 560.20
RPS improvement    : 1149.79%
